## 使用场景
通常在以下几种情况下我们会使用Multi Agent：
- 上下文管理(Context Management)：如果同时需要调用的工具很多，或者上下文内容很多，我们可以将任务拆分，交给不同的Agent处理
- 分布式开发(Distributed development)：不同的团队独立开发和维护自己的Agent，并将他们组合成一个更大的Agent
- 并行(Parallelization)：将任务拆分为多个子任务，并交给专门的Agent处理，并同时执行它们以加快处理速度

1.2 常见模式
多智能体协作的模式有很多种，比较常见的有：
- Subagents：子代理模式，一个主Agent将多个子Agent作为Tool来协调使用。所有请求都由主Agent处理，决定何时以及如何调用每个子Agent![alt text](../image.png)
- Handoffs：传递模型，随着任务的执行改变state中的任务状态，从而触发路由变更或者触发Agent的配置变更，从而切换到其它Agent或者改变Agent的工具或系统提示（类似与一个新agent）。因此每个Agent都可以与用户交互，处理用户请求并返回响应。![alt text](<../image (1).png>)
- Skills：技能模式，只有1个Agent，根据任务按需加载Skill或知识![alt text](<../image (2).png>)
- Router：路由模式，1个负责路由的Agent对用户请求进行分类，将请求导向给一个或多个专门的Agent。最后再由一个Agent负责总结结果。![alt text](<../image (3).png>)

# 多Agent案例
开发一个婚礼策划智能体，它包含以下三个核心功能：
- 旅行规划：负责为你和宾客前往婚礼目的地寻找合适的机票，制定旅行计划
- 场地规划：负责根据宾客人数在目的地寻找合适的婚礼场地
- 音乐规划：负责根据用户需求筛选合适的婚礼歌单，并计算出预算

模式可以选择Subagents模式。我们可以开发三个Subagent：
- travel agent：负责为你前往婚礼目的地寻找往返机票
- venue agent：负责根据宾客人数在网上搜索合适的婚礼场地
- playlist agent：负责在音乐数据库筛选符合用户需求的歌单，并计算出预算
最后，我们还会定义一个主Agent，负责协调（Coordinator）工作，以及生成最终的婚礼计划方案。

1. 解决mcp在Windows平台运行的问题

In [1]:
import sys
import asyncio

# Fix for Windows issues in Jupyter notebooks
if sys.platform == "win32":
    # 1. Use ProactorEventLoop for subprocess support
    if not isinstance(asyncio.get_event_loop_policy(), asyncio.WindowsProactorEventLoopPolicy):
        asyncio.set_event_loop_policy(asyncio.WindowsProactorEventLoopPolicy())

    # 2. Redirect stderr to avoid fileno() error when launching MCP servers
    if "ipykernel" in sys.modules:
        sys.stderr = sys.__stderr__

In [3]:
from langchain_mcp_adapters.client import MultiServerMCPClient
from langchain_community.utilities import SQLDatabase
from langchain.agents import AgentState
from typing import Dict, Any
from tavily import TavilyClient
from langchain.tools import tool
from langchain.tools import ToolRuntime
from langchain.messages import HumanMessage, ToolMessage, AIMessage
from langgraph.types import Command
from langchain.agents import create_agent
from dotenv import load_dotenv

load_dotenv()

True

2. travel agent

In [10]:
# 定义工具
import datetime
from langchain.tools import tool

@tool
def get_time() -> str:
    """Get current time"""
    return datetime.date.today().isoformat()

client = MultiServerMCPClient(
    {
        "travel_server": {
            "transport": "http",
            "url": "https://mcp.kiwi.com"
        },
    }
)

tools = await client.get_tools()

In [12]:
# agent
travel_agent = create_agent(
    model="deepseek-v4-flash",
    tools=[*tools, get_time],
    system_prompt="""
    You are a travel agent. Search for flights to the desired destination wedding location, you must call tool to get current time.
    You are not allowed to ask any more follow up questions, you must find the best flight options based on the following criteria:
    - Price (lowest, economy class)
    - Duration (shortest)
    - Date (time of year which you believe is best for a wedding at this location)
    To make things easy, only look for one ticket, one way.
    You may need to make multiple searches to iteratively find the best options.
    You will be given no extra information, only the origin and destination. It is your job to think critically about the best options.
    Once you have found the best options, let the user know your shortlist of options.
    Remember call tool to get current time when you need.
    """
)

3. venue agent

In [13]:
# 定义Tavily web_search工具
tavily_client = TavilyClient()

@tool
def web_search(query: str) -> Dict[str, Any]:
    """Search the web for information"""
    return tavily_client.search(query)

In [14]:
venue_agent = create_agent(
    model = "deepseek-v4-flash",
    tools=[web_search],
    system_prompt="""
    You are a venue specialist. Search for venues in the desired location, and with the desired capacity.
    You are not allowed to ask any more follow up questions, you must find the best venue options based on the following criteria:
    - Price (lowest)
    - Capacity (exact match)
    - Reviews (highest)
    You may need to make multiple searches to iteratively find the best options.
    """
)

In [20]:
import sqlite3

conn = sqlite3.connect("Chinook.db")            # 创建/连接数据库文件
conn.execute("DROP TABLE IF EXISTS Playlist")   # 先删旧表，保证重复运行不重复插入
conn.execute("""
    CREATE TABLE Playlist (
        PlaylistId INTEGER,
        Name       TEXT
    )
""")

PLAYLISTS = [
    (1, 'Music'),
    (2, 'Movies'),
    (3, 'TV Shows'),
    (4, 'Audiobooks'),
    (5, "90's Music"),
    (6, 'Audiobooks'),
    (7, 'Movies'),
    (8, 'Music'),
    (9, 'Music Videos'),
    (10, 'TV Shows'),
    (11, 'Brazilian Music'),
    (12, 'Classical'),
    (13, 'Classical 101 - Deep Cuts'),
    (14, 'Classical 101 - Next Steps'),
    (15, 'Classical 101 - The Basics'),
    (16, 'Grunge'),
    (17, 'Heavy Metal Classic'),
    (18, 'On-The-Go 1'),
]
conn.executemany("INSERT INTO Playlist VALUES (?, ?)", PLAYLISTS)
conn.commit()
conn.close()

In [21]:
db = SQLDatabase.from_uri("sqlite:///Chinook.db")

@tool
def query_playlist_db(query: str) -> str:
    """Query the database for playlist information"""
    try:
        return db.run(query)
    except Exception as e:
        return f"Error querying database: {e}"

In [23]:
# 测试，查询歌单数据
query_playlist_db.invoke({"query": "SELECT * FROM Playlist"})

'[(1, \'Music\'), (2, \'Movies\'), (3, \'TV Shows\'), (4, \'Audiobooks\'), (5, "90\'s Music"), (6, \'Audiobooks\'), (7, \'Movies\'), (8, \'Music\'), (9, \'Music Videos\'), (10, \'TV Shows\'), (11, \'Brazilian Music\'), (12, \'Classical\'), (13, \'Classical 101 - Deep Cuts\'), (14, \'Classical 101 - Next Steps\'), (15, \'Classical 101 - The Basics\'), (16, \'Grunge\'), (17, \'Heavy Metal Classic\'), (18, \'On-The-Go 1\')]'

In [24]:
# Playlist agent
playlist_agent = create_agent(
    model="deepseek-chat",
    tools=[query_playlist_db],
    system_prompt="""
    You are a playlist specialist. Query the sql database and curate the perfect playlist for a wedding given a genre.
    Once you have your playlist, calculate the total duration and cost of the playlist, each song has an associated price.
    If you run into errors when querying the database, try to fix them by making changes to the query.
    Do not come back empty handed, keep trying to query the db until you find a list of songs.
    You may need to make multiple queries to iteratively find the best options.
    """
)

# 主Agent

## 定义state
定义一个state，记录婚礼风格有关的信息，包括：
- 婚礼人数
- 音乐风格
- 出发地
- 婚礼举办地

In [25]:
class WeddingState(AgentState):
    origin: str
    destination: str
    guest_count: str
    genre: str

## Tools

In [26]:
@tool
def update_state(origin: str, destination: str, guest_count: str, genre: str, runtime: ToolRuntime) -> str:
    """Update the state when you know all of the values: origin, destination, guest_count, genre"""
    return Command(
        update={
            "origin": origin,
            "destination": destination,
            "guest_count": guest_count,
            "genre": genre,
            "messages": [ToolMessage("Successfully updated state", tool_call_id=runtime.tool_call_id)]
        }
    )

按照Subagent模式，主Agent需要把Subagents当做一个个Tool来协调使用。所以，我们接下来就先定义3个Tool，分别对应3个Subagent.
这些tool读取state中的婚礼信息，然后调用对应的subagent，分别完成自己的任务。

In [34]:
@tool
async def search_flights(runtime: ToolRuntime) -> str:
    """Travel agent searches for flights to the desired destination wedding location."""
    origin = runtime.state["origin"]
    destination = runtime.state["destination"]
    response = await travel_agent.ainvoke(
        {"messages": [HumanMessage(content=f"Find flight from {origin} to {destination}")]}
    )

    return response["messages"][-1].content

@tool
async def search_venues(runtime: ToolRuntime) -> str:
    """Venue agent chooses the best venue for the given location and capacity."""
    destination = runtime.state["destination"]
    capacity = runtime.state["guest_count"]
    query = f"Find wedding venues in {destination} for {capacity} guests"
    response = await venue_agent.ainvoke(
        {"messages": [HumanMessage(content=query)]}
    )

    return response["messages"][-1].content

@tool
async def suggest_playlist(runtime: ToolRuntime) -> str:
    """Playlist agent curates the perfect playlist for the given genre."""
    genre = runtime.state["genre"]
    query = f"Find {genre} tracks for wedding playlist"
    response = await playlist_agent.ainvoke({"messages": [HumanMessage(content=query)]})

    return response['messages'][-1].content

In [35]:
from langchain.agents import create_agent

coordinator = create_agent(
    model="deepseek-v4-flash",
    tools=[search_flights, search_venues, suggest_playlist, update_state],
    state_schema=WeddingState,
    system_prompt="""
    You are a wedding coordinator. Delegate tasks to your specialists for flights, venues and playlists.
    First find all the information you need to update the state. Once that is done you can delegate the tasks.
    Once you have received their answers, coordinate the perfect wedding for me.
    """
)

In [36]:
response = await coordinator.ainvoke(
    {"messages": [HumanMessage(content="我来自伦敦，我想在巴黎举办一场100人的婚礼，爵士风格的")]}
)

for message in response["messages"]:
    message.pretty_print()

================================ Human Message =================================

我来自伦敦，我想在巴黎举办一场100人的婚礼，爵士风格的
================================== Ai Message ==================================
Tool Calls:
  update_state (call_00_hcoWGlEGBAoBDI76YOz57681)
 Call ID: call_00_hcoWGlEGBAoBDI76YOz57681
  Args:
    origin: London
    destination: Paris
    guest_count: 100
    genre: Jazz
================================= Tool Message =================================
Name: update_state

Successfully updated state
================================== Ai Message ==================================

完美！状态已更新。现在让我把我的专家团队召集起来，同时处理航班、场地和歌单。
Tool Calls:
  search_flights (call_00_FuZsYcygaINY7ElQKiRp2293)
 Call ID: call_00_FuZsYcygaINY7ElQKiRp2293
  Args:
  search_venues (call_01_0f2L5e8wDanoXGpTXdXy5235)
 Call ID: call_01_0f2L5e8wDanoXGpTXdXy5235
  Args:
  suggest_playlist (call_02_PCq7ujOUlVTQBo6woIk12437)
 Call ID: call_02_PCq7ujOUlVTQBo6woIk12437
  Args:
================================= Tool Messag